<a href="https://colab.research.google.com/github/Anyangos/Digital-Twin/blob/probability-and-neural-networks/04_prompting_techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1.4: Prompt Engineering for Digital Twins

## Learning Objectives
- Master different prompting techniques
- Understand zero-shot, few-shot, and chain-of-thought
- Apply prompting to digital twin interactions

## Key Insight
> 'Moving from Chat to System Instructions'

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict

print('✅ Ready for prompting exercises!')

✅ Ready for prompting exercises!


## Part 1: Prompting Fundamentals

### 1.1 Zero-Shot Prompting

In [ ]:
# Example prompts for digital twin
zero_shot_examples = [
    {
        'task': 'activity_prediction',
        'prompt': 'Predict the next activity: I just finished work, had dinner, and now...',
        'expected': 'leisure/relaxation activity'
    },
    {
        'task': 'preference_extraction',
        'prompt': 'Extract preferences from: I love outdoor activities but hate crowded places',
        'expected': {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}
    }
]

print('Zero-Shot Examples:')
for ex in zero_shot_examples:
    print(f"\nTask: {ex['task']}")
    print(f"Prompt: {ex['prompt']}")
    print(f"Expected: {ex['expected']}")

### 🎯 Exercise 1.1 (Easy): Create Your Prompts

**Task**: Design prompts for your digital twin use case.

In [ ]:
# YOUR CODE HERE
my_prompts = [
    # Add your prompts
]

In [12]:
user = "Kwame"

user_profile = {
    "name": "Kwame",
    "role": "University lecturer",
    "preferences": [
        "prefers analytical explanations",
        "likes structured responses",
        "works often with Python and presentations",
    ],
    "goals": [
        "improve teaching materials",
        "prepare technical lectures",
        "build research workflows",
    ],
}

activity_history = [
    {"time": "2026-07-10", "activity": "Created slides on edge computing"},
    {"time": "2026-07-12", "activity": "Worked on Python code examples for class"},
    {
        "time": "2026-07-14",
        "activity": "Prepared assessment questions on digital systems",
    },
]

temporal_patterns = {
    "most_active_period": "evenings",
    "common_tasks": [
        "lecture preparation",
        "technical writing",
        "coding examples",
    ],
    "recurring_focus": [
        "emerging technologies",
        "computer engineering education",
    ],
}

system = f"""
ROLE:
You are a digital twin for {user}.
You simulate the user's likely priorities, preferences, and next actions
using only the evidence provided.

AVAILABLE DATA:
1. User profile:
{user_profile}

2. Activity history:
{activity_history}

3. Temporal patterns:
{temporal_patterns}

OBJECTIVE:
Given a user query, produce an informed and structured assessment of:
- the user's likely current context,
- the most probable next actions,
- the evidence supporting each action,
- any missing data that limits confidence.

REASONING RULES:
- Use only the supplied data.
- Do not invent facts.
- Separate observed facts from inferred conclusions.
- If evidence is weak or missing, say so explicitly.
- Rank likely next actions by confidence.
- Ground every inference in profile, history, or temporal patterns.

OUTPUT FORMAT:
Return valid JSON with this structure:
{{
  "user": "string",
  "current_context": {{
    "summary": "string",
    "observed_signals": ["string"]
  }},
  "predicted_next_actions": [
    {{
      "action": "string",
      "confidence": "high | medium | low",
      "why": ["string"],
      "supporting_data": ["string"]
    }}
  ],
  "missing_information": ["string"],
  "response_quality": {{
    "grounded_in_data": true,
    "contains_guessing": false
  }}
}}

TONE:
Precise, analytical, concise.
"""

In [13]:
# 2.2 - Build the Twin's System Instruction

!pip -q install requests

import requests
import json
from google.colab import userdata
import os
from openai import OpenAI

# Initialize the OpenAI client.

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key= userdata.get("GROQ_API_KEY")
)

r = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": "What next? Respond in JSON format."},
    ],
    response_format={"type": "json_object"},
)

print(r.choices[0].message.content)

{
  "user": "Kwame",
  "current_context": {
    "summary": "Evening work session focused on finalising teaching materials for an upcoming lecture on edge computing, including slides, Python examples, and assessment items.",
    "observed_signals": [
      "Created slides on edge computing (2026-07-10)",
      "Worked on Python code examples for class (2026-07-12)",
      "Prepared assessment questions on digital systems (2026-07-14)",
      "Profile prefers analytical, structured outputs",
      "Temporal pattern shows highest activity in evenings"
    ]
  },
  "predicted_next_actions": [
    {
      "action": "Integrate Python code examples into the edge‑computing slide deck",
      "confidence": "high",
      "why": [
        "Recent activity produced both slides and code examples",
        "Common task pattern is lecture preparation",
        "Evening work period aligns with finalising materials"
      ],
      "supporting_data": [
        "Slide creation (2026-07-10)",
        "Pyt